In [ ]:
import math
import torch
import torch.nn as nn
from torch.nn.functional import softmax
 
        
# Implementing the Multi-Head Attention
class SuspiciousMultiHeadAttention(nn.Module):
    def __init__(self, head_cnt, hidden_size):
        super().__init__()

        self.hidden_size = hidden_size
        self.heads = head_cnt  # Number of attention heads to use
        self.head_dim = hidden_size // head_cnt # Number of attention heads to use
         
        self.W_q = nn.Linear(hidden_size, hidden_size)  # Learned projection matrix for the queries
        self.W_k = nn.Linear(hidden_size, hidden_size)  # Learned projection matrix for the keys
        self.W_v = nn.Linear(hidden_size, hidden_size)  # Learned projection matrix for the values
        
        self.W_o = nn.Linear(hidden_size, hidden_size)  # Learned projection matrix for the multi-head output

        self.attn_do = nn.Dropout(0.1)
        self.out_do = nn.Dropout(0.1)
         
    def reshape_for_scores(self, x):
        bs, sl, ed = x.size()
        # x = x.view(x.size()[:2] + (self.heads, -1))
        x = x.view(bs, sl, self.heads, self.head_dim)
        x = x.permute(0, 2, 1, 3)
        return x
  
    def reshape_for_output(self, x):
        bs, sl, ed = x.size()
        x = x.permute(0, 2, 1, 3).contiguous()
        # x = x.reshape(x.size()[:2] +(self.d_k, ))
        x = x.view(bs, sl, ed)
        return x
  
    def forward(self, x, mask=None):
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)

        q_reshaped = self.reshape_for_scores(q)
        k_reshaped = self.reshape_for_scores(k)
        v_reshaped = self.reshape_for_scores(v)
         
        attention_scores = q_reshaped @ k_reshaped.transpose(1, 2) / math.sqrt(self.head_dim)
        
        attention_scores = self.attn_do(attention_scores)

        weights = softmax(attention_scores, dim=-1)
        if mask is not None:
            # weights = weights * mask
            weights = weights.masked_fill(mask == 0, float('-inf'))
            
        # forgot: @ v_reshaped
 
        output = self.reshape_for_output(attention_scores)
        return self.out_do(self.W_o(output))
     
     
hidden_dim = 128
head_cnt = 4
 
mha = SuspiciousMultiHeadAttention(head_cnt, hidden_dim)
 
n_tokens = 10
batch_size = 20
 
f = mha(torch.randn([batch_size, n_tokens, hidden_dim]))